[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C02/E02-1_Qualite_soudure_CORRIGE.ipynb)

# Qualification d'une soudure : fixer l'objectif et mettre le test de côté

## Le contexte

Une entreprise de chaudronnerie qualifie ses cordons de soudure après coup : contrôle visuel, puis examens complémentaires. Un cordon refusé se reprend, ce qui coûte du temps et de la matière. Le responsable qualité voudrait repérer, avant le contrôle, les cordons qui risquent d'être refusés, d'après ce que l'on sait de la soudure : procédé, matériau, soudeur, réglages, site.

Avant de modéliser, il faut fixer l'objectif : quelle grandeur prédire, avec quelles informations, jugée comment. Ici, on prédit la conformité d'un cordon (`conforme` ou `non-conforme`), avec les seules informations connues avant le contrôle, et on la juge par le F-score macro sur une base de test mise de côté dès le départ.

C'est un problème de classification binaire : la grandeur à prédire est une catégorie.

Défi : prédire `qualite_soudure` et obtenir un F-score macro supérieur à celui du modèle de départ sur la base de test. Ce modèle de départ n'utilise qu'une seule variable, le niveau de qualification du soudeur, et atteint un F-score macro de 0,647 : à vous d'ajouter les autres. La règle de découpage est fixée plus bas. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-Qualite_soudure.xlsx`, dans le dossier `donnees` du dépôt : 900 cordons, 19 colonnes. Une ligne est un cordon.

| Colonne | Signification | Unité |
|---|---|---|
| `id_soudure` | numéro du cordon | sans unité |
| `qualite_soudure` | cible : `conforme` ou `non-conforme` | catégorie |
| `niveau_qualification_soudeur` | `N1-Expert`, `N2-Intermédiaire` ou `N3-Junior` | catégorie |
| `type_soudage_procede` | `MIG/MAG` ou `TIG` | catégorie |
| `experience_soudeur_ans` | ancienneté du soudeur | années |
| `nb_passes_soudure` | nombre de passes | entier |
| `temperature_prechauffage_C` | température de préchauffage | °C |
| `intensite_courant_A` | intensité du courant | A |
| `type_assemblage` | `angle`, `bout_a_bout` ou `recouvrement` | catégorie |
| `epaisseur_tole_mm` | épaisseur de la tôle | mm |
| `materiau` | `acier carbone`, `acier inox` ou `aluminium` | catégorie |
| `gaz_protecteur_utilise` | gaz de protection employé ou non | vrai ou faux |
| `lot_production`, `poste_soudage` | lot et poste, renseignés pour une minorité de cordons | catégorie |
| `type_gaz_protection` | nature du gaz de protection | catégorie |
| `site_fabrication` | site de fabrication | catégorie |
| `controle_visuel_resultat` | résultat du contrôle visuel, `OK` ou `KO` | catégorie |
| `soudage_robotise` | soudage robotisé ou non | vrai ou faux |
| `vitesse_soudage_mm_min` | vitesse d'avance | mm/min |

Origine : jeu simulé, fabriqué par le script `generateurs/generer_qualite_soudure.py` du dépôt (graine 42). Les relations sont plausibles, sans être calées sur des essais réels ; ce support enseigne une méthode, et aucune conclusion de soudage n'en découle.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions et ses cases vides avant toute modélisation. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [1]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Qualite_soudure.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Qualite_soudure.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

print("dimensions :", df.shape)
vides = df.isna().sum()
print("cases vides par colonne :")
print(vides[vides > 0].to_string())
df.head(3)

dimensions : (900, 19)
cases vides par colonne :
experience_soudeur_ans    152
type_assemblage             4
lot_production            667
poste_soudage             667
type_gaz_protection       900
site_fabrication            3


,id_soudure,qualite_soudure,niveau_qualification_soudeur,type_soudage_procede,experience_soudeur_ans,nb_passes_soudure,temperature_prechauffage_C,intensite_courant_A,type_assemblage,epaisseur_tole_mm,materiau,gaz_protecteur_utilise,lot_production,poste_soudage,type_gaz_protection,site_fabrication,controle_visuel_resultat,soudage_robotise,vitesse_soudage_mm_min
0,1,conforme,N3-Junior,TIG,NaN,1,0,95,recouvrement,5,acier inox,True,NaN,NaN,NaN,Nantes,OK,True,202.7
1,2,conforme,N2-Intermédiaire,TIG,6.2,2,89,119,recouvrement,8,acier carbone,True,NaN,NaN,NaN,Le Havre,OK,False,198.3
2,3,non-conforme,N3-Junior,MIG/MAG,0.9,2,0,175,recouvrement,5,acier carbone,True,NaN,NaN,NaN,Le Havre,KO,False,378.4


Lecture du résultat : 900 lignes et 19 colonnes. Six colonnes ont des cases vides : `experience_soudeur_ans` (152), `type_assemblage` (4) et `site_fabrication` (3), `lot_production` (667), `poste_soudage` (667) et `type_gaz_protection`, vide pour les 900 cordons. Un lot ou un poste n'est renseigné que pour un cordon sur quatre environ.

### Étape 2 : fixer l'objectif et mettre le test de côté

Prédire, c'est utiliser des informations connues au moment de la décision pour deviner une grandeur connue plus tard. On regarde d'abord la cible, une colonne seule : la répartition des classes se lit sur tout le fichier. Puis un quart des cordons est écarté avant tout apprentissage, tout réglage et toute comparaison avec la cible. La graine 42 fixe le tirage : chacun obtient la même base de test, et les scores se comparent. Le tirage est stratifié : la proportion de cordons conformes est la même dans l'apprentissage et dans le test, ce qui compte quand une classe est moins fréquente que l'autre.

In [2]:
from sklearn.model_selection import train_test_split

cible = "qualite_soudure"
print(df[cible].value_counts().to_string())
print(f"part de cordons conformes : {(df[cible] == 'conforme').mean():.1%}")
X = df.drop(columns=cible)
y = df[cible]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=GRAINE, stratify=y)
print("apprentissage :", len(X_train), "cordons, dont", int((y_train == "conforme").sum()), "conformes")
print("test :", len(X_test), "cordons, dont", int((y_test == "conforme").sum()), "conformes")

qualite_soudure
non-conforme    543
conforme        357
part de cordons conformes : 39.7%
apprentissage : 675 cordons, dont 268 conformes
test : 225 cordons, dont 89 conformes


Lecture du résultat : 357 cordons conformes sur 900, soit 39,7 %. L'apprentissage garde 675 cordons et le test 225, avec la même proportion de conformes.

### Étape 3 : écarter ce qui n'est pas connu avant le contrôle

On se demande, colonne par colonne, si elle est connue au moment de la décision. Le résultat du contrôle visuel est suspect : on le confronte à la cible, sur l'apprentissage seul, car tout ce qui croise la cible se lit sans la base de test.

In [3]:
print(pd.crosstab(X_train["controle_visuel_resultat"], y_train))
colonnes_ecartees = ["id_soudure", "controle_visuel_resultat", "type_gaz_protection",
                     "lot_production", "poste_soudage"]
X_train = X_train.drop(columns=colonnes_ecartees)
X_test = X_test.drop(columns=colonnes_ecartees)
print("entrées gardées :", X_train.shape[1])

qualite_soudure           conforme  non-conforme
controle_visuel_resultat                        
KO                               0           407
OK                             268             0
entrées gardées : 13


Lecture du résultat : sur les 675 cordons d'apprentissage, le résultat du contrôle visuel coïncide avec la cible cordon par cordon : `OK` pour tous les conformes, `KO` pour tous les autres. C'est la cible sous un autre nom, connue seulement après le contrôle. L'utiliser comme entrée donnerait un modèle parfait, et inutilisable. Le numéro `id_soudure` n'a aucun sens physique, `type_gaz_protection` est vide, `lot_production` et `poste_soudage` ne sont renseignés que pour un cordon sur quatre : ces cinq colonnes sont écartées, les 13 autres restent des entrées possibles.

### Étape 4 : choisir la métrique avec un témoin

Une métrique se juge sur un modèle qui ne sait rien. Le témoin répond toujours la classe la plus fréquente de l'apprentissage. On compare l'exactitude (part de réponses justes) et le F-score macro, moyenne des F-scores des deux classes.

In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

temoin = DummyClassifier(strategy="most_frequent")
temoin.fit(X_train, y_train)
y_temoin = temoin.predict(X_test)
exactitude_temoin = accuracy_score(y_test, y_temoin)
f1_temoin = f1_score(y_test, y_temoin, average="macro")
print(f"exactitude du témoin : {exactitude_temoin:.3f}")
print(f"F-score macro du témoin : {f1_temoin:.3f}")

exactitude du témoin : 0.604
F-score macro du témoin : 0.377


Lecture du résultat : un témoin qui ne regarde aucune donnée a une exactitude de 0,604, ce qui paraît correct, et un F-score macro de 0,377. Le F-score macro punit un modèle qui ignore une classe : c'est la métrique du défi, et tout modèle doit d'abord battre ce témoin.

### Étape 5 : construire le modèle de départ

Le modèle de départ est volontairement simple : une régression logistique qui estime la probabilité d'un cordon conforme à partir d'une seule variable, le niveau de qualification du soudeur, qui est du texte à convertir en nombres. Il apprend sur `X_train` seul. Toutes les autres colonnes sont laissées de côté : elles sont votre marge de progression.

In [5]:
variable = "niveau_qualification_soudeur"

# Solution :
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

modele = make_pipeline(OneHotEncoder(), LogisticRegression())
modele.fit(X_train[[variable]], y_train)
# Fin de la solution

Pipeline(steps=[('onehotencoder', OneHotEncoder()),
                ('logisticregression', LogisticRegression())])

### Étape 6 : mesurer sur la base de test

Le modèle prédit les cordons qu'il n'a jamais vus. On lit le F-score macro, puis le détail par classe (précision et rappel) et la matrice de confusion, qui compte les bonnes et les mauvaises réponses.

In [6]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = modele.predict(X_test[[variable]])
score_depart = f1_score(y_test, y_pred, average="macro")
print(f"F-score macro : {score_depart:.3f}")
print(f"exactitude : {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred, digits=3))
matrice = confusion_matrix(y_test, y_pred, labels=["conforme", "non-conforme"])
print("matrice de confusion (lignes : réel, colonnes : prédit ; conforme, non-conforme)")
print(matrice)

F-score macro : 0.647
exactitude : 0.711
              precision    recall  f1-score   support

    conforme      0.800     0.360     0.496        89
non-conforme      0.692     0.941     0.798       136

    accuracy                          0.711       225
   macro avg      0.746     0.650     0.647       225
weighted avg      0.735     0.711     0.678       225

matrice de confusion (lignes : réel, colonnes : prédit ; conforme, non-conforme)
[[ 32  57]
 [  8 128]]


Lecture du résultat : avec le seul niveau de qualification, le F-score macro est de 0,647, au-dessus du témoin (0,377) : le niveau du soudeur renseigne, sans suffire. La matrice montre que le modèle, qui ne connaît que trois profils de soudeur, retrouve 32 cordons conformes sur 89 : il annonce « non-conforme » presque par défaut. C'est ce chiffre, mesuré sur des cordons que le modèle n'a jamais vus, que l'on a le droit d'annoncer.

## Pour aller plus loin : une démarche complète

Le modèle de départ de l'étape 5 reste le score de référence du défi. Les deux cellules suivantes montrent une démarche rigoureuse pour le dépasser. Les modèles candidats sont comparés par validation croisée sur l'apprentissage seul, sans toucher au test. On retient le plus simple des candidats dont le score reste à moins d'un écart-type du meilleur ; il est ensuite ajusté sur tout l'apprentissage et mesuré une seule fois sur le test. Les valeurs sont lues dans les sorties imprimées.

In [7]:
# Solution :
from sklearn.compose import make_column_transformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

numeriques = X_train.select_dtypes("number").columns.tolist()
categories = X_train.select_dtypes(exclude="number").columns.tolist()
preparation = make_column_transformer(
    (OneHotEncoder(handle_unknown="ignore"), categories),
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeriques))
candidats = {          # du plus simple au plus complexe
    "régression logistique": LogisticRegression(max_iter=1000),
    "arbre de profondeur 3": DecisionTreeClassifier(max_depth=3, random_state=GRAINE),
    "forêt aléatoire": RandomForestClassifier(n_estimators=300, random_state=GRAINE),
    "boosting par histogrammes": HistGradientBoostingClassifier(random_state=GRAINE)}
plis = StratifiedKFold(n_splits=5, shuffle=True, random_state=GRAINE)
moyennes, ecarts = {}, {}
for nom, estimateur in candidats.items():
    notes = cross_val_score(make_pipeline(preparation, estimateur), X_train, y_train,
                            cv=plis, scoring="f1_macro")
    moyennes[nom], ecarts[nom] = notes.mean(), notes.std()
    print(f"{nom} : F-score macro {notes.mean():.3f} +/- {notes.std():.3f} sur 5 plis")
meilleur = max(moyennes, key=moyennes.get)
seuil = moyennes[meilleur] - ecarts[meilleur]
nom_retenu = next(nom for nom in candidats if moyennes[nom] >= seuil)
print("meilleur score moyen :", meilleur)
print("modèle retenu, le plus simple à moins d'un écart-type du meilleur :", nom_retenu)
# Fin de la solution

régression logistique : F-score macro 0.772 +/- 0.016 sur 5 plis
arbre de profondeur 3 : F-score macro 0.682 +/- 0.031 sur 5 plis
forêt aléatoire : F-score macro 0.764 +/- 0.034 sur 5 plis
boosting par histogrammes : F-score macro 0.780 +/- 0.031 sur 5 plis
meilleur score moyen : boosting par histogrammes
modèle retenu, le plus simple à moins d'un écart-type du meilleur : régression logistique


In [8]:
# Solution :
modele_final = make_pipeline(preparation, candidats[nom_retenu])
modele_final.fit(X_train, y_train)
y_final = modele_final.predict(X_test)
f1_final = f1_score(y_test, y_final, average="macro")
print(f"{nom_retenu}, mesuré une fois sur le test : F-score macro {f1_final:.3f}")
print(f"gain sur le modèle de départ : {f1_final - score_depart:+.3f}")
print(classification_report(y_test, y_final, digits=3))

# Contre-épreuve : une colonne qui répète la cible donne un score trop beau pour être vrai.
entree = ["controle_visuel_resultat"]
fuite = make_pipeline(OneHotEncoder(), LogisticRegression())
fuite.fit(df.loc[X_train.index, entree], y_train)
f1_fuite = f1_score(y_test, fuite.predict(df.loc[X_test.index, entree]), average="macro")
print(f"avec le résultat du contrôle visuel en entrée : F-score macro {f1_fuite:.3f}")
print("lecture : un score parfait sur des données réelles signale presque toujours une fuite.")
# Fin de la solution

régression logistique, mesuré une fois sur le test : F-score macro 0.815
gain sur le modèle de départ : +0.168
              precision    recall  f1-score   support

    conforme      0.870     0.674     0.759        89
non-conforme      0.814     0.934     0.870       136

    accuracy                          0.831       225
   macro avg      0.842     0.804     0.815       225
weighted avg      0.836     0.831     0.826       225

avec le résultat du contrôle visuel en entrée : F-score macro 1.000
lecture : un score parfait sur des données réelles signale presque toujours une fuite.


## Le score de référence

Le score publié est le F-score macro du modèle de départ de l'étape 5 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [9]:
import hashlib

score = f1_score(y_test, modele.predict(X_test[[variable]]), average="macro")
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE f1_macro = %.3f" % score)

EMPREINTE_TEST = e4a7ada6277ee988660fa11f9ae44b84db09c32a94b3297192be1bb660d4c36c
SCORE_REFERENCE f1_macro = 0.647


## Synthèse

1. Fixer l'objectif, c'est choisir la cible, les informations disponibles au moment de la décision et la métrique, avant tout modèle.
2. Une colonne qui répète la cible, ou qui n'existe qu'après la décision, est écartée : sinon le score est excellent et le modèle inutilisable.
3. La base de test est mise de côté dès le départ, avec une graine fixée, et stratifiée quand une classe est plus rare : elle sert à mesurer, jamais à choisir.
4. Un témoin qui répond toujours la classe majoritaire fixe le seuil à battre : l'exactitude peut paraître bonne là où le F-score macro ne l'est pas.
5. Les modèles candidats se comparent par validation croisée sur l'apprentissage ; le test ne sert qu'une fois, pour le modèle retenu.